# Bloque 4 — Agentes de IA
## Notebook 00: De la llamada al modelo al equipo de agentes

---

### ¿Qué hemos hecho hasta ahora?

En los módulos anteriores hemos usado LLMs como **clasificadores**: le pasamos un texto, nos devuelve una etiqueta. El modelo no recuerda nada, no toma decisiones, no puede pedir más información. Es como un formulario que rellena solo.

```
[Texto] → LLM → [Categoría]
```

### ¿Qué es un agente?

Un **agente** es un LLM al que le añadimos tres capacidades:

| Capacidad | Qué significa |
|---|---|
| **Rol** | El modelo sabe quién es y cuál es su objetivo |
| **Memoria** | Puede recordar lo que ha descubierto |
| **Herramientas** | Puede llamar funciones externas (buscar en una BD, ejecutar código) |

El loop de un agente es:

```
┌─────────────────────────────────────────────┐
│                   AGENTE                    │
│                                             │
│  1. PERCIBE   → Lee la tarea/contexto       │
│  2. RAZONA    → Piensa qué hacer (CoT)      │
│  3. ACTÚA     → Llama herramienta / escribe │
│  4. OBSERVA   → Ve el resultado             │
│  5. Repite hasta completar la tarea         │
└─────────────────────────────────────────────┘
```

### ¿Qué es una crew (equipo)?

Una **crew** es un grupo de agentes especializados que colaboran. Cada uno tiene un rol concreto y sus outputs alimentan al siguiente:

```
[Investigador] → [Analista] → [Redactor] → Informe final
```

Esto es lo que vamos a construir en este bloque.

## Los tres primitivos de CrewAI

CrewAI organiza todo en tres conceptos:

### `Agent` — quién es
```python
Agent(
    role="Analista de amenazas",         # su título profesional
    goal="Identificar actores clave",     # su objetivo general
    backstory="Llevas 10 años en CTI...", # contexto que moldea su comportamiento
    llm=...,                              # el modelo que lo alimenta
)
```

### `Task` — qué tiene que hacer
```python
Task(
    description="Analiza estos perfiles y lista los 5 actores...", # instrucciones
    expected_output="Una lista numerada con nombre y justificación", # formato esperado
    agent=analista,                        # quién la ejecuta
)
```

### `Crew` — el equipo completo
```python
Crew(
    agents=[investigador, analista, redactor],
    tasks=[tarea1, tarea2, tarea3],
    process=Process.sequential,  # en orden, uno tras otro
)
resultado = crew.kickoff()        # ¡a trabajar!
```

In [1]:
# ─── VERIFICACIÓN DEL ENTORNO ────────────────────────────────────────────────
# Antes de empezar comprobamos que las librerías necesarias están instaladas.
# Si alguna falla aquí, revisar que has hecho `uv sync` en la raíz del repo.

import importlib

for libreria in ['crewai', 'ollama']:
    try:
        mod = importlib.import_module(libreria)
        version = getattr(mod, '__version__', 'desconocida')
        print(f'✓ {libreria} {version}')
    except ImportError:
        print(f'✗ {libreria} NO encontrado — ejecuta: uv sync')

✓ crewai 1.15.2
✓ ollama desconocida


In [2]:
# ─── IMPORTS ─────────────────────────────────────────────────────────────────
import os
import concurrent.futures

# CrewAI necesita una variable de entorno OPENAI_API_KEY para inicializarse,
# incluso cuando no usamos OpenAI. Le pasamos un valor falso — nunca se enviará
# a ningún servidor externo porque configuramos Ollama como backend local.
os.environ['OPENAI_API_KEY'] = 'NA'

# Los tres primitivos de CrewAI: el agente, la tarea y el equipo.
from crewai import Agent, Task, Crew, Process

# LLM es la clase que envuelve cualquier modelo de lenguaje para usarlo en CrewAI.
# Aquí lo apuntamos a nuestro Ollama local en lugar de a la API de OpenAI.
from crewai import LLM


def kickoff(crew, timeout=300):
    """Ejecuta crew.kickoff() en un hilo separado.

    CrewAI 1.x detecta el event loop de Jupyter/nbconvert y lanza un RuntimeError
    si se llama kickoff() directamente. Al correr en un hilo nuevo, el hilo no
    hereda el event loop activo y CrewAI puede crear el suyo sin conflicto.
    """
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff).result(timeout=timeout)


print('Imports correctos.')

Imports correctos.


In [3]:
# ─── CONFIGURACIÓN DEL LLM LOCAL ─────────────────────────────────────────────
# Creamos un objeto LLM que apunta a qwen2.5:14b corriendo en Ollama.
# El prefijo "ollama/" le indica a CrewAI que use el adaptador de LiteLLM para Ollama.
# base_url es la dirección donde escucha Ollama (por defecto el puerto 11434).

llm = LLM(
    model='ollama/qwen2.5:14b',
    base_url='http://localhost:11434',
)

# Comprobamos que Ollama responde haciendo una llamada mínima.
# Si aquí falla, asegúrate de que Ollama está corriendo: `ollama serve`
import ollama
respuesta_test = ollama.chat(
    model='qwen2.5:14b',
    messages=[{'role': 'user', 'content': 'Di solo: OK'}]
)
print('Ollama responde:', respuesta_test['message']['content'])

Ollama responde: I understand you've written "OK" in Diacritic Solresol, which is a constructed language that uses solfege syllables (do, re, mi, fa, sol, la, si) to convey messages. In Solresol, "DO SI" indeed means "yes" or "okay," so your message is affirmative or agreeable. Is there anything else you'd like to communicate in Solresol or any other way?


## Nuestro primer agente

Vamos a crear un agente muy sencillo: un analista de ciberseguridad al que le hacemos una pregunta sobre el grupo Conti. Sin datos externos, sin herramientas — solo el conocimiento que ya tiene el modelo.

In [4]:
# ─── DEFINIR EL AGENTE ───────────────────────────────────────────────────────

# El `role` es como el título en una tarjeta de visita: marca la especialidad.
# El `goal` es el objetivo que guía todas las decisiones del agente.
# El `backstory` da contexto que moldea el tono y profundidad de las respuestas.
# Cuanto más específico sea el backstory, más coherente será el comportamiento.

analista_conti = Agent(
    role='Analista de Threat Intelligence especializado en ransomware',
    goal='Proporcionar análisis precisos y útiles sobre grupos de ransomware '
         'basándose en los datos disponibles y el conocimiento del dominio.',
    backstory=(
        'Eres un analista senior de CTI (Cyber Threat Intelligence) con 8 años '
        'de experiencia rastreando grupos de ransomware. Has estudiado en '
        'profundidad el grupo Conti, sus métodos de operación y estructura interna. '
        'Respondes siempre en español, con precisión técnica y sin exagerar.'
    ),
    llm=llm,
    verbose=True,   # con verbose=True veremos el "pensamiento en voz alta" del agente
)

In [5]:
# ─── DEFINIR LA TAREA ────────────────────────────────────────────────────────

# La `description` es lo que el agente tiene que hacer, con todo el contexto necesario.
# El `expected_output` le dice al agente exactamente qué formato debe tener su respuesta.
# Sin expected_output, el agente puede dar respuestas demasiado largas o mal estructuradas.

tarea_intro = Task(
    description=(
        'Explica brevemente qué fue el grupo de ransomware Conti: '
        'cuándo operó, cómo estaba organizado y por qué fue significativo. '
        'Limita tu respuesta a lo que es factualmente verificable.'
    ),
    expected_output=(
        'Un párrafo de entre 100 y 150 palabras con: '
        '(1) período de actividad, (2) estructura organizativa, (3) impacto relevante.'
    ),
    agent=analista_conti,
)

In [6]:
# ─── CREAR Y LANZAR EL CREW ──────────────────────────────────────────────────

# El Crew es el contenedor que une agentes y tareas.
# Process.sequential ejecuta las tareas en el orden de la lista `tasks`.
# Con un solo agente y una sola tarea, es equivalente a una llamada directa al LLM,
# pero con la estructura que luego escalaremos a múltiples agentes.

crew_intro = Crew(
    agents=[analista_conti],
    tasks=[tarea_intro],
    process=Process.sequential,
    verbose=True,
)

# Usamos nuestra función kickoff() en vez de crew_intro.kickoff() directamente.
# kickoff() devuelve un objeto CrewOutput; .raw contiene el texto plano del resultado.
resultado = kickoff(crew_intro)

print('\n' + '='*60)
print('RESULTADO FINAL:')
print('='*60)
print(resultado.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b2d4a83d-16d4-49ae-9a7e-cb759211aa86                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explica brevemente qué fue el grupo de ransomware Conti: cuándo operó, cómo estaba organizado y por qué  │
│  fue significativo. Limita tu respuesta a lo que es factualmente verificable.                                   │
│  ID: ccc71841-0ae7-4dfd-a9da-f0d41d9d6453                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Threat Intelligence especializado en ransomware                                             │
│                                                                                                                 │
│  Task: Explica brevemente qué fue el grupo de ransomware Conti: cuándo operó, cómo estaba organizado y por qué  │
│  fue significativo. Limita tu respuesta a lo que es factualmente verificable.                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Threat Intelligence especializado en ransomware                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  El grupo Conti fue un importante sindicato criminal dedicado al ransomware que estuvo activo desde 2019 hasta  │
│  mediados de 2022. Fue conocido por su estructura jerárquica sofisticada, dividida en niveles como ejecutivos,  │
│  administración y operaciones tácticas. Este grupo utilizaba técnicas avanzadas de explotación para             │
│  comprometer redes corporativas y solicitar rescates multimillonarios. Conti fue significativo no solo por el   │
│  alcance de sus ataques —que incluyeron objetivos tan importantes como hospitales— sino también por su papel    │
│  en la proliferación del ransomware-as-a-service (RaaS), donde afiliados externos podían utilizar su            │
│  infraestructura para cometer crímenes cibernéticos a cambio de una parte de los beneficios.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Explica brevemente qué fue el grupo de ransomware Conti: cuándo operó, cómo estaba organizado y por qué  │
│  fue significativo. Limita tu respuesta a lo que es factualmente verificable.                                   │
│  Agent: Analista de Threat Intelligence especializado en ransomware                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


RESULTADO FINAL:
El grupo Conti fue un importante sindicato criminal dedicado al ransomware que estuvo activo desde 2019 hasta mediados de 2022. Fue conocido por su estructura jerárquica sofisticada, dividida en niveles como ejecutivos, administración y operaciones tácticas. Este grupo utilizaba técnicas avanzadas de explotación para comprometer redes corporativas y solicitar rescates multimillonarios. Conti fue significativo no solo por el alcance de sus ataques —que incluyeron objetivos tan importantes como hospitales— sino también por su papel en la proliferación del ransomware-as-a-service (RaaS), donde afiliados externos podían utilizar su infraestructura para cometer crímenes cibernéticos a cambio de una parte de los beneficios.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b2d4a83d-16d4-49ae-9a7e-cb759211aa86                                                                       │
│  Final Output: El grupo Conti fue un importante sindicato criminal dedicado al ransomware que estuvo activo     │
│  desde 2019 hasta mediados de 2022. Fue conocido por su estructura jerárquica sofisticada, dividida en niveles  │
│  como ejecutivos, administración y operaciones tácticas. Este grupo utilizaba técnicas avanzadas de             │
│  explotación para comprometer redes corporativas y solicitar rescates multimillonarios. Conti fue               │
│  significativo no solo por el alcance de sus ataques —que incluyeron objetivos tan importantes como             │
│  hospitales— sino también por su papel en la proliferación del ransomware-as-a-service (RaaS), donde afiliados  │
│  externos podían utilizar su infraestructura para cometer crímenes cibernéticos a cambio de una parte de los    │
│  beneficios.                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Anatomía del output

Cuando `verbose=True`, ves el loop interno del agente:

```
> Entering new AgentExecutor chain...
Thought: Tengo que explicar el grupo Conti...   ← el modelo razona
Final Answer: El grupo Conti fue...              ← la respuesta final
```

Esto es el **Chain of Thought** (CoT): el modelo "piensa en voz alta" antes de responder. Con `verbose=False` solo ves el resultado final — útil en producción, pero menos instructivo para aprender.

---

### Diferencia con una llamada directa a Ollama

```python
# Sin agentes (lo que hacíamos en bloque4):
respuesta = ollama.chat(model='qwen2.5:14b', messages=[{...}])

# Con agente:
crew.kickoff()  # El agente puede: recordar contexto, llamar herramientas, delegar
```

Con un solo agente y sin herramientas la diferencia es pequeña. La potencia viene cuando añadimos **múltiples agentes** que se pasan información entre ellos — eso lo vemos en el notebook 01.

In [7]:
# ─── EXPERIMENTO: verbose=False ──────────────────────────────────────────────
# Comparamos el output con verbose desactivado.
# El resultado es el mismo, pero sin ver el razonamiento intermedio.

analista_silencioso = Agent(
    role=analista_conti.role,
    goal=analista_conti.goal,
    backstory=analista_conti.backstory,
    llm=llm,
    verbose=False,   # <-- sin log de razonamiento
)

crew_silencioso = Crew(
    agents=[analista_silencioso],
    tasks=[
        Task(
            description=tarea_intro.description,
            expected_output=tarea_intro.expected_output,
            agent=analista_silencioso,
        )
    ],
    process=Process.sequential,
    verbose=False,
)

resultado2 = kickoff(crew_silencioso)
print('Resultado (sin verbose):')
print(resultado2.raw)

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Resultado (sin verbose):
El grupo Conti operó desde finales de 2019 hasta mediados de 2022 como uno de los principales actores del ransomware. Su estructura organizativa se basaba en una red jerárquica compuesta por líderes, sublíderes y miembros involucrados en actividades como ingeniería social, explotación de vulnerabilidades y negociaciones con sus víctimas. Conti fue significativo debido a su uso de tácticas sofisticadas y ciberdelitos coordinados a gran escala que afectaron a gobiernos, organismos gubernamentales y empresas en múltiples países.


## Resumen

| Concepto | Para qué sirve |
|---|---|
| `Agent` | Define quién es el modelo: rol, objetivo, personalidad |
| `Task` | Define qué tiene que hacer y qué formato debe tener el output |
| `Crew` | Une agentes y tareas; controla el orden de ejecución |
| `Process.sequential` | Ejecuta tareas en orden; el output de cada una pasa a la siguiente |
| `verbose=True` | Muestra el razonamiento interno — útil para depurar |

**Siguiente**: en el notebook `01_crew_investigacion.ipynb` construimos un equipo de 3 agentes que analiza datos reales de ContiLeaks.